In [0]:
# 1. Define the parameters (Default value, Widget Name, UI Label)
dbutils.widgets.text("years", "", "Years of data to ingest") # There are various widgets to choose from like Multiselect,etc
# 2. Get the string value from the workflow
years_str = dbutils.widgets.get("years")
# 3. Convert it to a Python list using list comprehension and split()
years = [int(year.strip()) for year in years_str.split(",") if year.strip()]

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
%pip install ../databricks_helpers
%pip install meteostat

In [0]:
from databricks_helpers.databricks_helper import DatabricksHelpers
import os
from datetime import datetime
import meteostat as ms
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pandas as pd
import time

dbh = DatabricksHelpers(dbutils, "aviation_analytics",spark)

ariports_table= "moyalanjintos_catalog.aviation_analytics.airports"
required_iata_view="moyalanjintos_catalog.aviation_analytics.req_airports_view"
bronze_flight_data = "moyalanjintos_catalog.aviation_analytics.bronze_flight_data"

meteo_dir = f"{dbh.volume_directory()}/meteo_data"
os.makedirs(meteo_dir, exist_ok=True)



In [0]:
#IATA_CODE to LAT, LONG taken from https://ourairports.com/data/

if not spark.catalog.tableExists(ariports_table):
    df=spark.read.csv(f"{dbh.volume_directory()}/airports.csv", header=True, inferSchema=True)
    df.filter(df.iata_code.isNotNull())\
    .withColumnsRenamed({"longitude_deg":"long","iata_code":"iata","latitude_deg":"lat"})\
    .drop("id","iso_region","municipality","scheduled_service","gps_code","local_code","home_link","wikipedia_link","keywords")\
    .write.mode("overwrite").saveAsTable(ariports_table)
    spark.sql(f"create view if not exists {required_iata_view} as select * from {ariports_table} where iata in (select distinct ORIGIN from {bronze_flight_data})")



In [0]:
weather_list = []
required_locations = [(row.iata, (row.lat, row.long)) for row in spark.table(required_iata_view).select("iata","lat","long").collect()]

In [0]:
monthly_limit = 500
batch_size = 3

# Convert input list to Spark DataFrame
df = spark.createDataFrame(required_locations, ["iata_code", "coords"])
#Split to lat long
df = df.withColumn("lat", col("coords._1")) \
       .withColumn("lon", col("coords._2")).drop("coords")


In [0]:
import pyspark.sql.functions as F
from datetime import datetime

output_schema = """
    time TIMESTAMP,
    temp DOUBLE,
    tmin DOUBLE,
    tmax DOUBLE,
    rhum DOUBLE,
    prcp DOUBLE,
    snwd DOUBLE,
    wspd DOUBLE,
    wpgt DOUBLE,
    pres DOUBLE,
    tsun DOUBLE,
    cldc DOUBLE,
    iata_code STRING,
    station STRING
"""

batch_size = 4

# 🔥 Databricks-safe existence check
def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except:
        return False

# Find missing years
missing_years = []
for year in years:
    year_dir = f"{meteo_dir}/meteo_weather_data_{year}"
    if not path_exists(year_dir):
        missing_years.append(year)

# 🔥 Optimize: prepare df ONCE
df_prepared = df.limit(monthly_limit).repartition(2)

def fetch_weather_pandas(iterator):
    import meteostat as ms
    import pandas as pd
    import time

    expected_cols = [
        'time','temp','tmin','tmax','rhum','prcp',
        'snwd','wspd','wpgt','pres','tsun','cldc','iata_code'
    ]

    for pdf in iterator:
        results = []
        requests_made = 0

        for _, row in pdf.iterrows():
            try:
                point = ms.Point(row['lat'], row['lon'])
                station=ms.stations.nearby(point, limit=1)
                # ✅ use global start/end (set per loop)
                print(f"{start}/{end}")
                data = ms.daily(station, start, end).fetch().reset_index()

                if not data.empty:
                    data['iata_code'] = row['iata_code']
                    # data = data.reindex(columns=expected_cols)
                    results.append(data)

                requests_made += 1

                if requests_made % batch_size == 0:
                    time.sleep(1)

            except Exception as e:
                print(f"Error fetching data for {row.get('iata_code', 'Unknown')}: {e}")

        if results:
            final_pdf = pd.concat(results, ignore_index=True)
            # 🔥 This removed chunked array issue of arrow, below code forces pd to use numpy array
            final_pdf = pd.DataFrame({
                col: final_pdf[col].to_numpy()
                for col in final_pdf.columns
            })
            yield final_pdf
        else:
            empty_df = pd.DataFrame(columns=expected_cols)
            empty_df = pd.DataFrame({
                col: empty_df[col].to_numpy()
                for col in empty_df.columns
            })
            yield empty_df

In [0]:
# 🚀 MAIN LOOP (only change you needed)
for year in missing_years:
    print(f"Processing year: {year}")

    start = datetime(year, 1, 1)
    end = datetime(year, 12, 31)# by pythons late binding it works

    weather_df = df_prepared.mapInPandas(fetch_weather_pandas, schema=output_schema) #it requires schema
   # weather_df = df_prepared.mapInPandas(fetch_weather_pandas)
#todo:- schema evolution need to check

    output_path = f"{meteo_dir}/meteo_weather_data_{year}"
    weather_df.write.mode("overwrite").json(output_path)

    print(f"Saved: {output_path}")